In [ ]:
# Video Content Model for 360-degree video (tiles + layers)
# This code defines a VideoContentModel class suitable for integration into a Gymnasium environment.
# It partitions a 360° equirectangular video into tiles and quality layers, computes per-tile sizes,
# and provides helpers to map a viewport (yaw, pitch, FOV) to tile indices.
#
# Demo at the bottom shows usage and prints results.
from dataclasses import dataclass, field
import numpy as np
from typing import List, Tuple, Dict, Iterable

In [ ]:
@dataclass
class VideoContentModel:
    duration_s: float                # total video duration in seconds
    fps: float                       # frames per second
    tile_rows: int                   # number of latitude (pitch) divisions
    tile_cols: int                   # number of longitude (yaw) divisions
    n_layers: int                    # number of quality layers (0..n_layers-1)
    layer_bitrates_kbps: List[float] # per-layer bitrate per tile in kbps (list length == n_layers)
    segment_duration_s: float = 2.0  # duration of each segment (s) used for paging/prefetch
    projection: str = "equirectangular" # currently only 'equirectangular' supported
    layer_multipliers: List[float] = field(default_factory=list)  # optional multipliers per layer to modify size
    
    def __post_init__(self):
        assert len(self.layer_bitrates_kbps) == self.n_layers, "layer_bitrates_kbps length must match n_layers"
        if not self.layer_multipliers:
            # default: each layer is 1.0 multiplier, but higher layers may be larger resolution -> larger size
            self.layer_multipliers = [1.0 + 0.5*L for L in range(self.n_layers)]
        
        assert len(self.layer_multipliers) == self.n_layers
        
        # derived values
        self.n_frames = int(np.ceil(self.duration_s * self.fps))
        self.frames_per_segment = max(1, int(np.round(self.segment_duration_s * self.fps)))
        self.n_segments = int(np.ceil(self.n_frames / self.frames_per_segment))

        # tile angular size (equirectangular)
        self.tile_width_deg = 360.0 / self.tile_cols
        self.tile_height_deg = 180.0 / self.tile_rows
    
    def num_tiles(self) -> int:
        return self.tile_rows * self.tile_cols
    
    def tile_bbox(self, row: int, col: int) -> Tuple[float, float, float, float]:
        """
        Returns tile bounding box in (yaw_min, yaw_max, pitch_min, pitch_max) in degrees.
        yaw in [-180, 180), pitch in [-90, 90].
        row=0 is top (pitch near +90?), we'll set row 0 -> pitch max (90 - small)
        """
        # pitch ranges from +90 (top) to -90 (bottom); we index rows top->bottom
        pitch_max = 90.0 - row * self.tile_height_deg
        pitch_min = pitch_max - self.tile_height_deg
        yaw_min = -180.0 + col * self.tile_width_deg
        yaw_max = yaw_min + self.tile_width_deg
        return (yaw_min, yaw_max, pitch_min, pitch_max)
    
    @staticmethod
    def normalize_yaw(yaw: float) -> float:
        # normalize to [-180,180)
        y = ((yaw + 180.0) % 360.0) - 180.0
        return y
    
    def viewport_bbox(self, yaw_center: float, pitch_center: float, fov_h: float, fov_v: float) -> Tuple[float, float, float, float]:
        """
        Returns viewport bounding box (yaw_min, yaw_max, pitch_min, pitch_max).
        Handles yaw wrap-around.
        """
        yaw_center = self.normalize_yaw(yaw_center)
        yaw_half = fov_h / 2.0
        yaw_min = yaw_center - yaw_half
        yaw_max = yaw_center + yaw_half
        # keep pitch within [-90,90]
        pitch_min = max(-90.0, pitch_center - fov_v / 2.0)
        pitch_max = min(90.0, pitch_center + fov_v / 2.0)
        return (yaw_min, yaw_max, pitch_min, pitch_max)
    
    def _yaw_interval_overlap(self, a_min, a_max, b_min, b_max) -> bool:
        """
        Determine overlap for intervals on circular yaw axis (degrees).
        We'll map both to ranges and check overlap with wrap-around handling.
        """
        # normalize to [-180,180)
        a_min = self.normalize_yaw(a_min)
        a_max = self.normalize_yaw(a_max)
        b_min = self.normalize_yaw(b_min)
        b_max = self.normalize_yaw(b_max)
        
        def intervals_to_list(x_min, x_max):
            # returns list of non-wrapping intervals (min,max) in [-180,180)
            if x_min <= x_max:
                return [(x_min, x_max)]
            else:
                # wraps across -180/180
                return [(x_min, 180.0), (-180.0, x_max)]
        
        a_parts = intervals_to_list(a_min, a_max)
        b_parts = intervals_to_list(b_min, b_max)
        for am, aM in a_parts:
            for bm, bM in b_parts:
                # check overlap
                if not (aM <= bm or bM <= am):
                    return True
        return False
    
    def tile_intersects_viewport(self, row: int, col: int, yaw_center: float, pitch_center: float, fov_h: float, fov_v: float) -> bool:
        y_min, y_max, p_min, p_max = self.tile_bbox(row, col)
        v_ymin, v_ymax, v_pmin, v_pmax = self.viewport_bbox(yaw_center, pitch_center, fov_h, fov_v)
        yaw_overlap = self._yaw_interval_overlap(y_min, y_max, v_ymin, v_ymax)
        pitch_overlap = not (p_max <= v_pmin or p_min >= v_pmax)
        return yaw_overlap and pitch_overlap
    
    def tiles_for_viewport(self, yaw_center: float, pitch_center: float, fov_h: float, fov_v: float) -> List[Tuple[int,int]]:
        """
        Returns list of (row, col) tiles that intersect the viewport.
        """
        tiles = []
        for r in range(self.tile_rows):
            for c in range(self.tile_cols):
                if self.tile_intersects_viewport(r, c, yaw_center, pitch_center, fov_h, fov_v):
                    tiles.append((r,c))
        return tiles
    
    def tile_index(self, row: int, col: int) -> int:
        return row * self.tile_cols + col
    
    def index_to_rowcol(self, idx: int) -> Tuple[int,int]:
        r = idx // self.tile_cols
        c = idx % self.tile_cols
        return r, c
    
    def tile_size_bytes(self, layer: int, segment_duration_s: float = None) -> int:
        """
        Compute the size in bytes for one tile at a given layer for a segment of segment_duration_s.
        We use per-layer bitrate per tile (kbps) and multiplier to scale.
        """
        if segment_duration_s is None:
            segment_duration_s = self.segment_duration_s
        kbps = self.layer_bitrates_kbps[layer] * self.layer_multipliers[layer]
        bytes_per_second = (kbps * 1000.0) / 8.0
        size = int(np.round(bytes_per_second * segment_duration_s))
        return size
    
    def tiles_payload_for_viewport(self, yaw_center: float, pitch_center: float, fov_h: float, fov_v: float,
                                   selected_layer: int, segment_duration_s: float = None) -> Dict:
        """
        Returns a dict with per-tile sizes (bytes) for tiles intersecting the viewport, for one segment.
        Also returns totals.
        """
        tiles = self.tiles_for_viewport(yaw_center, pitch_center, fov_h, fov_v)
        if segment_duration_s is None:
            segment_duration_s = self.segment_duration_s
        per_tile_size = self.tile_size_bytes(selected_layer, segment_duration_s)
        details = []
        for r,c in tiles:
            details.append({"row": r, "col": c, "index": self.tile_index(r,c), "size_bytes": per_tile_size})
        total = sum(d["size_bytes"] for d in details)
        return {"tiles": details, "total_bytes": total, "n_tiles": len(details), "layer": selected_layer}
    
    def segment_time_bounds(self, segment_idx: int) -> Tuple[float,float]:
        start = segment_idx * self.segment_duration_s
        end = min(self.duration_s, (segment_idx + 1) * self.segment_duration_s)
        return (start, end)
    
    def tiles_for_segment_given_viewport(self, segment_idx: int, yaw_center: float, pitch_center: float, fov_h: float, fov_v: float, layer:int):
        # wrapper combining segment and tiles_for_viewport
        return {"segment": segment_idx, "time_bounds": self.segment_time_bounds(segment_idx),
                **self.tiles_payload_for_viewport(yaw_center, pitch_center, fov_h, fov_v, layer)}
    
    def summary(self) -> Dict:
        return {
            "duration_s": self.duration_s,
            "fps": self.fps,
            "n_frames": self.n_frames,
            "n_segments": self.n_segments,
            "segment_duration_s": self.segment_duration_s,
            "n_tiles": self.num_tiles(),
            "tile_grid": (self.tile_rows, self.tile_cols),
            "n_layers": self.n_layers
        }


In [ ]:
if __name__ == "__main__":
    # Create a model: 60s, 30fps, 6x12 tiles, 3 layers
    model = VideoContentModel(
        duration_s=60.0,
        fps=30.0,
        tile_rows=6,
        tile_cols=12,
        n_layers=3,
        layer_bitrates_kbps=[100.0, 300.0, 900.0],  # approximate kbps per tile per layer
        segment_duration_s=2.0
    )
    
    print("Video content summary:")
    for k,v in model.summary().items():
        print(f"  {k}: {v}")
    print()
    
    # Example viewport: facing yaw=10°, pitch=0°, horizontal FOV 90°, vertical FOV 90°
    yaw_c, pitch_c, fov_h, fov_v = 10.0, 0.0, 90.0, 90.0
    print(f"Viewport center yaw={yaw_c}°, pitch={pitch_c}°, FOV_h={fov_h}°, FOV_v={fov_v}°")
    tiles = model.tiles_for_viewport(yaw_c, pitch_c, fov_h, fov_v)
    print(f"Tiles intersecting viewport (row,col): {tiles}")
    print(f"Number of tiles: {len(tiles)}")
    print()
    
    # Compute payload for each layer for segment 0
    for layer in range(model.n_layers):
        payload = model.tiles_payload_for_viewport(yaw_c, pitch_c, fov_h, fov_v, selected_layer=layer)
        print(f"Layer {layer}: n_tiles={payload['n_tiles']}, total_bytes={payload['total_bytes']:,} (~{payload['total_bytes']/1000:.1f} KB)")
    
    # Example: tiles for a wrapped viewport near yaw=178° which crosses the -180/180 boundary
    yaw_wrap = 178.0
    tiles_wrap = model.tiles_for_viewport(yaw_wrap, pitch_c, fov_h=60.0, fov_v=60.0)
    print()
    print(f"Tiles for wrapped viewport yaw={yaw_wrap}° (fov_h=60): {tiles_wrap}")
    
    # Example: get tiles for segment 5 at layer 2
    seg5 = model.tiles_for_segment_given_viewport(5, yaw_c, pitch_c, fov_h, fov_v, layer=2)
    print()
    print("Segment 5 tiles payload:", seg5)